# 🧠 LUOKAI Neural Brain — Training Notebook

Train LUOKAI's neural brain on a free Google Colab GPU/CPU.

**What this does:**
1. Clones the LuoOS repo
2. Generates (or expands) the training corpus
3. Trains the NumPy transformer for as many steps as you want
4. Lets you download `luokai_brain.npz` to drop back into your repo

**Honest expectations:** the pure-NumPy brain does not use the GPU
(NumPy is CPU-only). A Colab CPU core runs ~5-7 steps/sec. For a
genuinely fluent brain, run 30,000-50,000 steps — about 1.5-3 hours.
The training checkpoints every 500 steps, so you can stop and the
`.npz` is always up to date.

If you want GPU-speed training, see the optional PyTorch port at the
bottom of this notebook.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/luokai25/luo_os-v_0.1.git luo_os
%cd luo_os
!pip -q install numpy

## 2. (Optional) Expand the corpus

The bigger and more varied the corpus, the more fluent the brain.
Edit `luokai/neural/corpus.py` to add more conversation pairs, then
regenerate. Or just use the default 360-pair starter corpus.

In [ ]:
# Regenerate the corpus (more repeats = more training signal)
!python -m luokai.neural.corpus --repeats 10
!echo '---'
!wc -l luokai/neural/corpus.txt

## 3. Train

Run this cell as many times as you like — each run **resumes** from
the last checkpoint. 5,000 steps per run is a comfortable chunk.

In [ ]:
# Train 5000 steps (resumes automatically if a checkpoint exists)
!python -m luokai.neural.train --steps 5000

In [ ]:
# Run again for another 5000 — repeat until loss flattens (~0.1-0.3)
!python -m luokai.neural.train --steps 5000

## 4. Test the brain

In [ ]:
import sys; sys.path.insert(0, '.')
from luokai.neural.infer import NeuralBrain
nb = NeuralBrain()
print('Stats:', nb.stats())
print()
for msg in ['hi', 'who are you', 'open my files', 'what can you do', 'thanks']:
    print(f'  {msg!r:25s} -> {nb.respond(msg, temperature=0.4)!r}')

## 5. Export a shippable brain & download it

This saves an inference-only `.npz` (no optimizer state, much smaller).

In [ ]:
from luokai.neural.brain import LuokaiBrain
from pathlib import Path
b = LuokaiBrain.load()
out = Path('luokai/neural/luokai_brain.npz')
b.save_inference(out)
print(f'Saved {out} — {out.stat().st_size/1024/1024:.1f} MB, {b.steps} steps')

# Download both the brain and the vocab
from google.colab import files
import shutil
shutil.copy(Path.home()/'.luo_os'/'neural'/'vocab.json', 'luokai/neural/vocab.json')
files.download('luokai/neural/luokai_brain.npz')
files.download('luokai/neural/vocab.json')

## 6. Drop the trained brain into your repo

Copy the downloaded `luokai_brain.npz` and `vocab.json` into your local
`luo_os/luokai/neural/` folder, commit, and push. LUOKAI will load the
trained brain automatically on next start.

```bash
cp ~/Downloads/luokai_brain.npz  luo_os/luokai/neural/
cp ~/Downloads/vocab.json        luo_os/luokai/neural/
cd luo_os && git add luokai/neural/ && git commit -m 'trained brain' && git push
```